# Entregable 3 · Opción B: piloto y entrenamiento de NAGFuse-B (protocolo v4)

Este cuaderno corre la **opción B** autorizada por el docente. No modifica nada de la versión A: usa colas, carpetas y marcas distintas (`NAGFuse-B1`, `NAGFuse-B2`, `NAGFuse-B3` y `NAGFuse-B`).

**Tres fases, en este orden (ver `entregable3/experiments/protocolo_v4_opcion_B.md`):**

| Fase | `REF` | `FASE` | Qué hace | Tiempo aprox. en A100 |
|---|---|---|---|---|
| 0. Prueba de humo | `e3-protocolo-v4-piloto` | `"humo"` | Todo el camino con 48 imágenes y 1–2 épocas, en carpetas `_smoke` | 15–25 min |
| 1. Piloto | `e3-protocolo-v4-piloto` | `"piloto"` | Entrena los 3 candidatos **solo con validación** (semilla 100, 30 épocas máx.) y aplica la regla de elección | 20–35 min |
| 2. Confirmatoria | `e3-protocolo-v4` | `"confirmatoria"` | 20 semillas del candidato elegido, con evaluación en prueba, robustez y conjuntos adicionales | 2.5–3.5 h |

Entre la fase 1 y la 2 hay un paso **fuera de Colab**: fijar `ELEGIDO_B` en `e3/cfg.py`, anotar el piloto en `design_log.md`, completar el protocolo v4 y crear el tag `e3-protocolo-v4`. **El conjunto de prueba no se toca hasta la fase 2.**

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo* → **A100** (más rápida y, en total, más barata en unidades que L4 o T4 para este trabajo; ver el MD de traspaso). Drive con al menos 5 GB libres.

## 1. Parámetros de esta sesión

In [ ]:
FASE = "humo"               # "humo" -> "piloto" -> "confirmatoria"
REF = "e3-protocolo-v4-piloto" if FASE in ("humo", "piloto") else "e3-protocolo-v4"
WORKER_ID = "0"
HORAS = 11                  # límite para lanzar tareas nuevas (la tarea en curso termina)
ACCUM = 1                   # 2 si hay error de memoria de GPU (el lote efectivo sigue en 32)
REPO_URL = "https://github.com/Anyita-12/deep-learning-colaborativo.git"
COLA = {"humo": "b_piloto", "piloto": "b_piloto", "confirmatoria": "b"}[FASE]

## 2. Drive, repositorio y dependencias

In [ ]:
import os, subprocess, sys, json, signal
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/deep-learning-colaborativo")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", f"+refs/tags/{REF}:refs/tags/{REF}"], check=False)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REF], check=False)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-f", "FETCH_HEAD"], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Commit en uso:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "entregable3" / "requirements-train.txt")], check=True)
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
assert torch.cuda.is_available(), "Activa una GPU en Entorno de ejecución → Cambiar tipo de entorno"

E3 = REPO / "entregable3"
os.chdir(E3)
os.environ["PYTHONPATH"] = str(E3)
os.environ["TRASHNET_ACCUM"] = str(ACCUM)
os.environ["E3_WORK"] = "/content/e3_work"
os.environ["E3_DRIVE"] = "/content/drive/MyDrive/E3_TrashNet"
os.environ["E3_COLA"] = COLA                     # cola de la opción B
os.environ["E3_MIRROR_EVERY_S"] = "3600"          # espejo de checkpoints en Drive solo en corridas de más de 1 h
Path(os.environ["E3_DRIVE"]).mkdir(parents=True, exist_ok=True)
(Path(os.environ["E3_DRIVE"]) / f"commit_en_uso_B_{FASE}.txt").write_text(f"{REF} {commit}\n")
if FASE == "confirmatoria":
    sys.path.insert(0, str(E3)); from e3 import cfg as _c
    assert _c.ELEGIDO_B, "El tag e3-protocolo-v4 no tiene ELEGIDO_B fijado: revisa el paso entre fases."
    print("Candidato elegido:", _c.ELEGIDO_B)

## 3. Credenciales de Kaggle (solo si faltan los conjuntos adicionales en Drive)

In [ ]:
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Credenciales de Kaggle cargadas desde Secrets.")
except Exception as e:
    kj = Path(os.environ["E3_DRIVE"]) / "kaggle.json"
    if kj.exists():
        k = json.loads(kj.read_text()); os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = k["username"], k["key"]
        print("Credenciales de Kaggle cargadas desde Drive/E3_TrashNet/kaggle.json.")
    else:
        print("Sin credenciales de Kaggle: GD v2 y WaRP-C quedarán pendientes (el entrenamiento sigue).", e)

## 4. Pruebas unitarias (incluyen los tres candidatos de la opción B)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=E3)
assert r.returncode == 0, "Las pruebas fallaron: no lances el entrenamiento."

## 5. Ejecución

- `humo`: prueba de humo completa en carpetas `_smoke`. Si dice **PRUEBA DE HUMO SUPERADA**, cambia `FASE = "piloto"` y vuelve a ejecutar todo.
- `piloto`: entrena los tres candidatos (solo validación) y al final imprime la tabla y la elección.
- `confirmatoria`: 20 semillas de NAGFuse-B.

Si la sesión se cae, vuelve a ejecutar todo con la misma `FASE`: lo terminado se salta.

In [ ]:
def lanzar(*args):
    p = subprocess.Popen([sys.executable, "-m", "e3.orchestrator", *args], cwd=E3, env=dict(os.environ),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    (Path("/content") / "orquestador.pid").write_text(str(p.pid))
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, signal.SIGTERM)
        print("\nDetenido. El progreso hasta el último checkpoint está en Drive.")
    return p.wait()

if FASE == "humo":
    rc = lanzar("smoke")
    print("Código de salida:", rc, "→", "PRUEBA SUPERADA: cambia FASE a \"piloto\"" if rc == 0 else "REVISA LOS ERRORES")
else:
    assert lanzar("setup") == 0
    assert lanzar("prepare", "--worker", WORKER_ID) == 0
    lanzar("run", "--worker", WORKER_ID, "--hours", str(HORAS))
    if FASE == "piloto":
        lanzar("piloto")

## 6. Detener (solo si hace falta)

In [ ]:
pid_file = Path("/content/orquestador.pid")
if pid_file.exists():
    try:
        os.killpg(int(pid_file.read_text()), signal.SIGTERM); print("Orquestador detenido.")
    except ProcessLookupError:
        print("No había un orquestador en ejecución.")

## 7. Estado y resumen del piloto (en cualquier momento)

In [ ]:
subprocess.run([sys.executable, "-m", "e3.orchestrator", "status"], cwd=E3, env=dict(os.environ))
if FASE in ("humo", "piloto"):
    subprocess.run([sys.executable, "-m", "e3.orchestrator", "piloto"], cwd=E3, env=dict(os.environ))

## 8. Evaluación pendiente en conjuntos adicionales (solo si alguno quedó pendiente)

In [ ]:
# lanzar("extra-pass", "--worker", WORKER_ID)